In [ ]:
from pathlib import Path

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the project root by looking for README.md and data/."""
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "data").exists():
            return path

    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT.relative_to(PROJECT_ROOT) if PROJECT_ROOT != PROJECT_ROOT else PROJECT_ROOT.name)
print(PROCESSED_DIR.relative_to(PROJECT_ROOT) if PROCESSED_DIR != PROJECT_ROOT else PROJECT_ROOT.name)
print(FIGURES_DIR.relative_to(PROJECT_ROOT) if FIGURES_DIR != PROJECT_ROOT else PROJECT_ROOT.name)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pred_path = PROCESSED_DIR / 'sleep_staging_cv_predictions_mini.csv'
pred_df = pd.read_csv(pred_path)

print(pred_df.shape)
pred_df.head()

In [ ]:
print(pred_df['subject_id'].value_counts())
print()
print(pred_df['stage_label'].value_counts())
print()
print(pred_df['predicted_stage'].value_counts())

In [ ]:
def compute_fragmentation_metrics(stage_sequence, epoch_duration_sec=30):
    seq = pd.Series(stage_sequence).reset_index(drop=True)

    n_epochs = len(seq)
    total_minutes = n_epochs * epoch_duration_sec / 60
    total_hours = n_epochs * epoch_duration_sec / 3600

    if n_epochs == 0:
        return {
            'n_epochs': 0,
            'total_minutes': 0,
            'total_hours': 0,
            'n_transitions': 0,
            'transitions_per_hour': np.nan,
            'wake_epochs': 0,
            'wake_minutes': 0,
            'wake_pct': np.nan,
            'wake_intrusions': 0,
            'fragmentation_index': np.nan
        }

    transitions = (seq != seq.shift()).sum() - 1
    transitions = max(transitions, 0)

    wake_epochs = (seq == 'W').sum()
    wake_minutes = wake_epochs * epoch_duration_sec / 60
    wake_pct = wake_epochs / n_epochs * 100

    wake_mask = (seq == 'W').astype(int)
    wake_starts = ((wake_mask == 1) & (wake_mask.shift(fill_value=0) == 0)).sum()
    wake_intrusions = int(wake_starts)

    transitions_per_hour = transitions / total_hours if total_hours > 0 else np.nan
    fragmentation_index = (transitions + wake_intrusions) / total_hours if total_hours > 0 else np.nan

    return {
        'n_epochs': n_epochs,
        'total_minutes': total_minutes,
        'total_hours': total_hours,
        'n_transitions': int(transitions),
        'transitions_per_hour': transitions_per_hour,
        'wake_epochs': int(wake_epochs),
        'wake_minutes': wake_minutes,
        'wake_pct': wake_pct,
        'wake_intrusions': wake_intrusions,
        'fragmentation_index': fragmentation_index
    }

In [ ]:
rows = []

for subject_id, sub_df in pred_df.groupby('subject_id'):
    sub_df = sub_df.sort_values('epoch_start_sec').copy()

    true_metrics = compute_fragmentation_metrics(sub_df['stage_label'])
    pred_metrics = compute_fragmentation_metrics(sub_df['predicted_stage'])

    row = {'subject_id': subject_id}

    for k, v in true_metrics.items():
        row[f'true_{k}'] = v

    for k, v in pred_metrics.items():
        row[f'pred_{k}'] = v

    rows.append(row)

fragmentation_df = pd.DataFrame(rows)
fragmentation_df

In [ ]:
fragmentation_df['diff_transitions'] = fragmentation_df['pred_n_transitions'] - fragmentation_df['true_n_transitions']
fragmentation_df['diff_wake_intrusions'] = fragmentation_df['pred_wake_intrusions'] - fragmentation_df['true_wake_intrusions']
fragmentation_df['diff_fragmentation_index'] = fragmentation_df['pred_fragmentation_index'] - fragmentation_df['true_fragmentation_index']

fragmentation_df

In [ ]:
frag_path = PROCESSED_DIR / 'fragmentation_metrics_mini.csv'
fragmentation_df.to_csv(frag_path, index=False)

print(frag_path)
print(frag_path.exists())

In [ ]:
plot_df = fragmentation_df.copy()
x = np.arange(len(plot_df))

plt.figure(figsize=(8, 5))
plt.bar(x - 0.2, plot_df['true_n_transitions'], width=0.4, label='True')
plt.bar(x + 0.2, plot_df['pred_n_transitions'], width=0.4, label='Predicted')

plt.xticks(x, plot_df['subject_id'])
plt.ylabel('Number of transitions')
plt.title('True vs Predicted Stage Transitions')
plt.legend()
plt.tight_layout()

fig_path = FIGURES_DIR / 'fragmentation_true_vs_pred_transitions_mini.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(fig_path)
print(fig_path.exists())

In [ ]:
plot_df = fragmentation_df.copy()
x = np.arange(len(plot_df))

plt.figure(figsize=(8, 5))
plt.bar(x - 0.2, plot_df['true_wake_intrusions'], width=0.4, label='True')
plt.bar(x + 0.2, plot_df['pred_wake_intrusions'], width=0.4, label='Predicted')

plt.xticks(x, plot_df['subject_id'])
plt.ylabel('Wake intrusions')
plt.title('True vs Predicted Wake Intrusions')
plt.legend()
plt.tight_layout()

fig_path = FIGURES_DIR / 'fragmentation_true_vs_pred_wake_intrusions_mini.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(fig_path)
print(fig_path.exists())

In [ ]:
plot_df = fragmentation_df.copy()
x = np.arange(len(plot_df))

plt.figure(figsize=(8, 5))
plt.bar(x - 0.2, plot_df['true_fragmentation_index'], width=0.4, label='True')
plt.bar(x + 0.2, plot_df['pred_fragmentation_index'], width=0.4, label='Predicted')

plt.xticks(x, plot_df['subject_id'])
plt.ylabel('Fragmentation index')
plt.title('True vs Predicted Fragmentation Index')
plt.legend()
plt.tight_layout()

fig_path = FIGURES_DIR / 'fragmentation_index_true_vs_pred_mini.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(fig_path)
print(fig_path.exists())

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    fragmentation_df['true_fragmentation_index'],
    fragmentation_df['pred_fragmentation_index']
)

for _, row in fragmentation_df.iterrows():
    plt.annotate(
        row['subject_id'],
        (row['true_fragmentation_index'], row['pred_fragmentation_index'])
    )

min_val = min(fragmentation_df['true_fragmentation_index'].min(), fragmentation_df['pred_fragmentation_index'].min())
max_val = max(fragmentation_df['true_fragmentation_index'].max(), fragmentation_df['pred_fragmentation_index'].max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')
plt.xlabel('True fragmentation index')
plt.ylabel('Predicted fragmentation index')
plt.title('True vs Predicted Fragmentation Index')
plt.tight_layout()

fig_path = FIGURES_DIR / 'fragmentation_index_scatter_mini.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(fig_path)
print(fig_path.exists())

In [ ]:
summary_cols = [
    'subject_id',
    'true_n_transitions',
    'pred_n_transitions',
    'true_wake_intrusions',
    'pred_wake_intrusions',
    'true_fragmentation_index',
    'pred_fragmentation_index',
    'diff_fragmentation_index'
]

fragmentation_df[summary_cols]

## Fragmentation interpretation

This notebook extends the sleep staging pipeline by deriving subject-level sleep fragmentation metrics from both true and predicted stage sequences.

The main goal is not only to classify sleep stages, but also to assess whether model outputs preserve clinically meaningful sleep stability patterns such as:
- stage transitions
- wake intrusions
- fragmentation burden

## Conclusion

The fragmentation analysis shows how epoch-level stage predictions can be translated into higher-level sleep stability metrics.

Even when stage-level classification is imperfect, predicted stage sequences may still retain useful subject-level information about fragmentation structure. This is important for neurotech applications where downstream sleep-quality summaries can be as valuable as raw stage predictions.

## Interpretation

The fragmentation analysis demonstrates that stage-level predictions can be converted into higher-level sleep stability summaries. This is important because many neurotech and sleep-tech applications rely not only on per-epoch staging, but also on interpretable subject-level sleep quality metrics.